# Mitra — Telco Churn

Mitra é um modelo tabular fundacional da Amazon (2025), baseado em Transformer com ~72M parâmetros,
pré-treinado em dados sintéticos via in-context learning. Disponível no AutoGluon 1.4+.

São comparados dois modos:
- **Zero-shot**: sem fine-tuning, inferência direta via ICL.
- **Fine-tuned**: ajuste fino dos pesos no dataset de treino.

O melhor modo (KS no val) é usado para avaliação final no teste.

> **Instalação:** `uv add "autogluon.tabular[mitra]"`

In [ ]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
from sklearn import metrics as skm
from importlib.metadata import version as pkg_version
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from autogluon.tabular import TabularPredictor
from model_utils import load_data, compute_metrics, save_results, print_scores

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
AG_VERSION = pkg_version('autogluon.tabular')
print(f'AutoGluon version: {AG_VERSION}')

## 1. Carregar dados

In [2]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

yvl_np  = y_val.values.astype(int)
yte_np  = y_test.values.astype(int)

LABEL = 'Churn'

# AutoGluon exige DataFrame com coluna label incluída
train_df = X_train.copy(); train_df[LABEL] = y_train.values
val_df   = X_val.copy();   val_df[LABEL]   = y_val.values

print(f'Churn rate treino: {y_train.mean():.3f}')

Train: (7242, 22)  Val: (1057, 22)  Test: (1057, 22)
Churn rate treino: 0.500


## 2. Modo Zero-Shot (sem fine-tuning)

In [ ]:
os.makedirs('results/mitra/ag_zeroshot', exist_ok=True)

predictor_zs = TabularPredictor(
    label=LABEL,
    problem_type='binary',
    eval_metric='roc_auc',
    path='results/mitra/ag_zeroshot',
    verbosity=1,
)
predictor_zs.fit(
    train_df,
    hyperparameters={"MITRA": {"fine_tune": False, "ag.max_memory_usage_ratio": 2}},
    fit_weighted_ensemble=False,
)

probs_zs_val = predictor_zs.predict_proba(X_val)[1].values
fpr, tpr, _ = skm.roc_curve(yvl_np, probs_zs_val)
ks_zs = float(np.max(tpr - fpr))
print(f'Zero-shot  →  KS (val): {ks_zs:.4f}')

## 3. Modo Fine-Tuned — Optuna HPO

Optuna busca `fine_tune_steps`, `lr`, `patience` e `warmup_steps` maximizando KS no val (igual MLP/KAN/XGB).
10 trials — cada trial é mais pesado que modelos torch (carrega 72M params).

In [ ]:
def objective(trial):
    steps   = trial.suggest_int('fine_tune_steps', 50, 200)
    lr      = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    patience = trial.suggest_categorical('patience', [30, 40, 50])
    warmup  = trial.suggest_categorical('warmup_steps', [500, 1000, 1500])

    path = f'results/mitra/ag_trial_{trial.number}'
    os.makedirs(path, exist_ok=True)
    pred = TabularPredictor(
        label=LABEL,
        problem_type='binary',
        eval_metric='roc_auc',
        path=path,
        verbosity=0,
    )
    pred.fit(
        train_df,
        hyperparameters={"MITRA": {
            "fine_tune": True,
            "fine_tune_steps": steps,
            "lr": lr,
            "patience": patience,
            "warmup_steps": warmup,
            "ag.max_memory_usage_ratio": 2,
        }},
        fit_weighted_ensemble=False,
    )
    probs_val = pred.predict_proba(X_val)[1].values
    fpr, tpr, _ = skm.roc_curve(yvl_np, probs_val)
    ks = float(np.max(tpr - fpr))
    trial.set_user_attr('predictor', pred)
    trial.set_user_attr('ks', ks)
    print(f'  trial {trial.number:>2d}  steps={steps:>3d}  lr={lr:.2e}  patience={patience}  warmup={warmup}  KS={ks:.4f}')
    return ks

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=10, show_progress_bar=False)

best_trial = study.best_trial
best_ft = {
    'ks':        best_trial.value,
    'predictor': best_trial.user_attrs['predictor'],
    'params':    best_trial.params,
}
print(f'\nMelhor fine-tuned  →  KS={best_ft["ks"]:.4f}  params={best_ft["params"]}')

## 4. Selecionar melhor modo e avaliar no teste

In [ ]:
if ks_zs >= best_ft['ks']:
    best_mode = 'zero-shot'
    best_predictor = predictor_zs
    print(f'Modo selecionado: zero-shot  (KS val={ks_zs:.4f})')
else:
    p = best_ft['params']
    best_mode = f'fine-tuned (steps={p["fine_tune_steps"]} lr={p["lr"]:.2e})'
    best_predictor = best_ft['predictor']
    print(f'Modo selecionado: {best_mode}  (KS val={best_ft["ks"]:.4f})')

probs = best_predictor.predict_proba(X_test)[1].values
preds = (probs >= 0.5).astype(int)

scores = compute_metrics(yte_np, preds, probs)
print('\nScores no teste:')
print_scores(scores)

os.makedirs('results/mitra', exist_ok=True)
joblib.dump({'mode': best_mode, 'params': best_ft.get('params'), 'scores': scores}, 'results/mitra/model.pkl')

save_results('mitra', yte_np, preds, probs, scores)
print(f'\nNota: Mitra ({best_mode}), AutoGluon {AG_VERSION}.')